# 00 · AWS Bedrock credentials and settings

Every model in these notebooks is called through AWS Bedrock. There are no API keys: LiteLLM signs each request with the AWS credentials of your SSO profile, through boto3's normal credential chain. Nothing secret is stored in a notebook or in `.env.local`.

**One-time setup**

```bash
cd notebooks
cp .env.local.example .env.local         # set AWS_PROFILE to a profile with Bedrock access
aws sso login --profile <that profile>   # IAM Identity Center profiles; again when the session expires
# or: aws login --profile <that profile> # console-login profiles (needs botocore[crt], in the notebooks group)
```

`.env.local` (git-ignored) holds settings only:

```bash
AWS_PROFILE=<profile name from ~/.aws/config>
AWS_REGION=us-west-2
# MODEL_FAST / MODEL_SMART / MODEL_ALT=bedrock/<model or inference profile ID>   (optional)
LIVE=0        # 1 = real Bedrock calls, real cost
```

**How the loading works** ([`_env.py`](_env.py))

- `import _env` reads `.env.local` with `python-dotenv` into `os.environ`. boto3 picks up `AWS_PROFILE`, and LiteLLM reads `AWS_REGION`, so no notebook passes credentials or a region.
- `_env.py` refuses to load if `.env.local` contains `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` or `AWS_SESSION_TOKEN`. Long-lived access keys are not allowed; use a profile with temporary credentials.
- A variable already exported in your shell wins over the file.
- Model strings are `bedrock/<ID>`. The defaults use `us.` cross-region inference profiles, which newer Claude models on Bedrock need for on-demand use. Override them in `.env.local` if your account has different models enabled.
- Change `.env.local`, then **restart the kernel**: the file is read once per kernel.

**Keeping notebooks clean**

- `status()` prints whether `AWS_PROFILE` is set, the region and the models. It never prints the profile name, because SSO profile names often contain the AWS account ID. Don't print `os.environ`, run `%env`/`!env`, or print a `get_caller_identity()` result (it contains your account ID): output is saved inside the `.ipynb`.
- After a `LIVE=1` run, clear the outputs (*Edit → Clear Outputs of All Cells*) before committing, or commit only offline runs.

In [1]:
from _env import status

status()

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


Live check (skipped offline). It confirms the profile resolves to working credentials, without printing the identity, then lists which configured models your account can use in this region. Both calls are free. The last loop makes one tiny paid call per model.

In [2]:
import os

import boto3
import litellm
from _env import LIVE, MODELS

litellm.suppress_debug_info = True

if not LIVE:
    print("offline: skipped (set LIVE=1 in .env.local and restart the kernel)")
else:
    session = boto3.Session(region_name=os.environ["AWS_REGION"])
    session.client("sts").get_caller_identity()  # raises if the login session expired
    print(f"credentials ok (region {session.region_name})")

    bedrock = session.client("bedrock")
    available = {p["inferenceProfileId"]
                 for page in bedrock.get_paginator("list_inference_profiles").paginate()
                 for p in page["inferenceProfileSummaries"]}
    available |= {m["modelId"] for m in bedrock.list_foundation_models()["modelSummaries"]}
    for model in MODELS:
        model_id = model.removeprefix("bedrock/").removeprefix("converse/")
        print(f"{model:45} {'available' if model_id in available else 'NOT FOUND in this region'}")

    print()
    for model in MODELS:
        try:
            r = litellm.completion(model=model, max_tokens=512,
                                   messages=[{"role": "user", "content": "Reply with OK."}])
            print(f"{model:45} ok: {r.choices[0].message.content!r}")
        except litellm.APIError as e:
            print(f"{model:45} {type(e).__name__}: {str(e)[:80]}")

credentials ok (region us-west-2)


bedrock/us.anthropic.claude-haiku-5-5         available
bedrock/us.anthropic.claude-opus-5-5          available
bedrock/us.amazon.nova-2-lite-v1:0            available



bedrock/us.anthropic.claude-haiku-5-5         ok: 'OK'


bedrock/us.anthropic.claude-opus-5-5          ok: 'OK'


bedrock/us.amazon.nova-2-lite-v1:0            ok: 'OK.'
